# 🤖 LeStudio — LeRobot Training on Colab

This notebook trains a [LeRobot](https://github.com/huggingface/lerobot) policy using a config uploaded from **LeStudio**.

### How to use
1. **Run cells 1–3** to install dependencies and log in to Hugging Face.
2. In **LeStudio → Train tab**, click **`Train on Colab`** — it uploads your config to the Hub and copies a snippet to your clipboard.
3. **Cell 4에 스니펫 붙여넣기** → 실행하면 config 로드.
4. **Run cell 5** to start training.

> 💡 Make sure the Colab runtime has a GPU attached: **Runtime → Change runtime type → T4 GPU**

In [ ]:
# ⚙️ Cell 1 — Install dependencies (conflict-safe)
# Run once per session. Takes ~3-5 minutes on fresh runtimes.
import importlib.metadata as im
import subprocess, sys

def _pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

# Keep Hugging Face stack compatible with current LeRobot ecosystem.
_pip("--upgrade", "pip")
_pip("lerobot", "huggingface_hub<1.0", "transformers<5")

# Colab often preinstalls torchaudio that mismatches torch.
torch_base = im.version("torch").split("+", 1)[0]
_pip("--upgrade", f"torchaudio=={torch_base}")

print("✅ Installed")
print("torch       :", im.version("torch"))
print("torchaudio  :", im.version("torchaudio"))
print("transformers:", im.version("transformers"))
print("hf_hub      :", im.version("huggingface_hub"))

In [ ]:
# 🔑 Cell 2 — Hugging Face login
# Enter your HF token (needs read access to your dataset repo).
# Get one at https://huggingface.co/settings/tokens
from huggingface_hub import login
login()

In [ ]:
# 🔧 Cell 3 — LeStudio config helper (do not modify)
import json, os
from huggingface_hub import hf_hub_download

def lestudio_load_config(repo_id: str, config_path: str = "lestudio_train_config.json") -> dict:
    token = os.getenv("HF_TOKEN") or os.getenv("HUGGINGFACE_HUB_TOKEN")
    cfg_file = hf_hub_download(
        repo_id=repo_id,
        filename=config_path,
        repo_type="dataset",
        token=token,
        force_download=True,
    )
    with open(cfg_file, "r", encoding="utf-8") as f:
        cfg = json.load(f)
    print(f"✅ Config loaded from {repo_id}/{config_path}")
    print(f"   policy     : {cfg.get('policy')}")
    print(f"   dataset    : {cfg.get('dataset_repo')}")
    print(f"   steps      : {cfg.get('steps')}")
    print(f"   batch_size : {cfg.get('batch_size')}")
    print(f"   device     : {cfg.get('train_device')}")
    if cfg.get('output_repo'):
        print(f"   output_repo: {cfg.get('output_repo')}")
    return cfg

In [ ]:
# 📋 Cell 4 — LeStudio snippet
# ─────────────────────────────────────────────────────────
# LeStudio → Train tab → 'Train on Colab' 클릭 후
# 복사된 코드를 이 셀 전체에 붙여넣고 실행하세요.
# Paste the copied snippet here and run this cell.
# ─────────────────────────────────────────────────────────

In [ ]:
# 🚀 Cell 5 — Start training
import collections
import importlib.metadata as im
import json
import os
import subprocess
import sys

import torch
from huggingface_hub import HfApi, HfFolder, hf_hub_download

# policy type mapping (lerobot CLI uses 'tdmpc' for tdmpc2)
policy_raw = str(cfg.get("policy", "act"))
policy = "tdmpc" if policy_raw == "tdmpc2" else policy_raw

dataset_repo = str(cfg.get("dataset_repo", "")).strip()
steps        = int(cfg.get("steps", 50000))
device       = str(cfg.get("train_device", "cuda")).strip() or "cuda"
batch_size   = cfg.get("batch_size")
lr           = cfg.get("lr")
output_repo  = cfg.get("output_repo")

if not dataset_repo:
    raise ValueError("dataset_repo is empty. Re-run Cell 4 (LeStudio snippet) first.")

if device.lower().startswith("cuda") and not torch.cuda.is_available():
    print("⚠️ CUDA runtime is not available in this Colab session. Falling back to CPU.")
    device = "cpu"

cmd = [
    sys.executable, "-m", "lerobot.scripts.lerobot_train",
    f"--policy.type={policy}",
    f"--dataset.repo_id={dataset_repo}",
    f"--steps={steps}",
    f"--policy.device={device}",
    "--policy.push_to_hub=false",
]

if batch_size:
    cmd.append(f"--batch_size={int(batch_size)}")
if lr:
    cmd.append(f"--optimizer.lr={float(lr)}")
if output_repo:
    cmd += [f"--hub_repo_id={output_repo}", "--policy.push_to_hub=true"]

# extra overrides from LeStudio (advanced)
extra_overrides = [str(o) for o in cfg.get("extra_overrides", [])]
if not any(o.startswith("--num_workers=") for o in extra_overrides):
    extra_overrides.append("--num_workers=2")
for override in extra_overrides:
    cmd.append(override)

env = os.environ.copy()
hf_token = env.get("HF_TOKEN") or env.get("HUGGINGFACE_HUB_TOKEN") or HfFolder.get_token()
if hf_token:
    env["HF_TOKEN"] = hf_token
    env["HUGGINGFACE_HUB_TOKEN"] = hf_token

def ensure_dataset_codebase_tag(repo_id, token):
    api = HfApi(token=token)
    info_path = hf_hub_download(
        repo_id=repo_id,
        filename="meta/info.json",
        repo_type="dataset",
        token=token,
        force_download=True,
    )
    with open(info_path, "r", encoding="utf-8") as f:
        info = json.load(f)

    version = str(info.get("codebase_version", "")).strip()
    if not version:
        raise RuntimeError("meta/info.json is missing codebase_version.")

    refs = api.list_repo_refs(repo_id, repo_type="dataset")
    tags = {t.name for t in getattr(refs, "tags", [])}
    if version in tags:
        print(f"✅ Dataset tag exists: {version}")
        return

    if not token:
        raise RuntimeError(
            f"Missing dataset tag '{version}' and no HF token is available to create it."
        )

    print(f"ℹ️ Creating missing dataset tag: {version}")
    try:
        api.create_tag(repo_id=repo_id, tag=version, repo_type="dataset")
    except Exception as e:
        if "already exists" not in str(e).lower():
            raise
    print(f"✅ Dataset tag ready: {version}")

try:
    ensure_dataset_codebase_tag(dataset_repo, hf_token)
except Exception as e:
    print(f"❌ Dataset preflight failed: {e}")
    raise

def run_with_live_logs(command, env_vars):
    tail = collections.deque(maxlen=120)
    proc = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env_vars,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="")
        tail.append(line.rstrip("\n"))
    return proc.wait(), list(tail)

print("▶ Running:", " ".join(cmd))
print()

returncode, log_tail = run_with_live_logs(cmd, env)
if returncode == 0:
    print("\n✅ Training complete!")
else:
    print(f"\n❌ Training exited with code {returncode}")
    if log_tail:
        print("\n--- Last log lines ---")
        for line in log_tail[-40:]:
            print(line)

    print("\n--- Package versions ---")
    for name in ["lerobot", "torch", "torchaudio", "transformers", "huggingface_hub"]:
        try:
            print(f"{name:15s}: {im.version(name)}")
        except Exception:
            print(f"{name:15s}: (not installed)")